# SigAlg's `Operators.cov` method

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `Operators.cov` method in SigAlg is a method for computing *covariances* of random variables, both unconditional and conditional versions. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.Operators.cov).

## Mathematical definition

Let $X,Y:\Omega \to \mathbb{R}$ be two random variables on a probability space $(\Omega, \mathcal{F},P)$ for which $E(X^2), E(Y^2) < \infty$, and let $\mathcal{G}$ be a sub-$\sigma$-algebra of $\mathcal{F}$. The *conditional covariance* of $X$ and $Y$ with respect to $\mathcal{G}$ is any $\mathcal{G}$-measurable random variable $\sigma(X, Y \mid \mathcal{G})$ for which

$$
\sigma(X,Y\mid \mathcal{G}) = E(XY \mid \mathcal{G}) - E(X\mid \mathcal{G})E(Y\mid \mathcal{G}).
$$

In the case that $\Omega$ is finite (as it always is, in SigAlg), the $\sigma$-algebra $\mathcal{G}$ is determined by its (finitely many) atoms, and the space $L^2(\Omega, \mathcal{G}, P)$ has an orthogonal basis given by the indicator functions of the atoms of $\mathcal{G}$ with nonzero probability. Then we have

$$
\sigma(X,Y\mid \mathcal{G}) = \sum_B \sigma(X|_B, Y|_B) I_B,
$$

where the sum extends over all atoms $B$ of $\mathcal{G}$ with nonzero probability, and where $\sigma(X|_B, Y|_B)$ is the covariance of the restricted random variables $X|_B, Y|_B:B\to \mathbb{R}$ where $B$ is equipped with the conditional probability measure $P_B$ such that $P_B(C) = P(C)/P(B)$ for $C\subset B$.

## API examples


### Unconditional covariances

We begin by defining a sample space $\Omega = \{0,1,2,3,4\}$ and a probability measure $P$ on $\Omega$.

In [28]:
import numpy as np

from sigalg.core import ProbabilityMeasure, SampleSpace

rng = np.random.default_rng(42)

Omega = SampleSpace().from_sequence(size=5)
P = ProbabilityMeasure(sample_space=Omega).from_rand(random_state=rng)
print(P)

Probability measure 'P':
        probability
sample             
0          0.320930
1          0.311850
2          0.318334
3          0.037349
4          0.011538


Define two random variables $X,Y: \Omega \to \mathbb{R}$ on the sample space $\Omega$ and set their `probability_measure` attribute to $P$ so that all covariances will be computed relative to $P$.

In [29]:
from sigalg.core import RandomVariable

X = RandomVariable(domain=Omega).from_randint(low=-20, high=21, random_state=rng)
Y = RandomVariable(domain=Omega, name="Y").from_randint(
    low=-10, high=11, random_state=rng
)
X.prob_measure = P
Y.prob_measure = P

print(X, "\n")
print(Y)

Random variable 'X':
         X
sample    
0        1
1       20
2       10
3       11
4        9 

Random variable 'Y':
        Y
sample   
0       6
1       0
2      -8
3       7
4      -1


All covariances in SigAlg are instances of `RandomVariable`. The unconditional covariance `cov(X, Y)` is thus a constant random variable whose value is the usual covariance $\sigma(X,Y) = E(XY) - E(X)E(Y)$. The `item` method extracts $\sigma(X,Y)$ from `cov(X, Y)`.

In [30]:
from sigalg.core import Operators

cov = Operators.cov

covar_rv = cov(X, Y)
covar = cov(X, Y).item()

print(covar_rv, "\n")
print(covar)

Random variable 'cov(X, Y)':
        cov(X, Y)
sample           
0      -16.962212
1      -16.962212
2      -16.962212
3      -16.962212
4      -16.962212 

-16.962211857983537


### Conditional covariances

Define a $\sigma$-algebra $\mathcal{G}$ on $\Omega$ with atoms $A_0=\{0,1\}$ and $A_1 = \{2,3,4\}$. 

In [31]:
from sigalg.core import SigmaAlgebra

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 1,
    }
)

Compute the conditional covariance $\sigma(X,Y\mid \mathcal{G})$.

In [32]:
print(cov(X, Y, G))

Random variable 'cov(X, Y|G)':
        cov(X, Y|G)
sample             
0        -28.494132
1        -28.494132
2          1.182969
3          1.182969
4          1.182969


### Testing properties of covariances

#### Conditional covariances are linear combinations

We noted in the definition that the conditional covariance may be expressed as a linear combination of the indicator functions of the atoms of the $\sigma$-algebra. In the next code cell, we test this. Notice that the output matches the output above.

In [33]:
I = RandomVariable.indicator_of

linear_combo = sum([cov(X(A), Y(A)).item() * I(A) for A in G.to_atoms()])

print(linear_combo.with_name("linear_combo"))

Random variable 'linear_combo':
        linear_combo
sample              
0         -28.494132
1         -28.494132
2           1.182969
3           1.182969
4           1.182969


#### Alternate formula for covariances

The conditional covariance may equivalently be defined as

$$
\sigma(X , Y \mid \mathcal{G}) = E \left[ (X- E(X\mid \mathcal{G})) (Y-E(Y\mid \mathcal{G})) \mid \mathcal{G}\right].
$$

We check this equation in the next code cell.

In [34]:
E = Operators.expectation

covar = cov(X, Y, G)
expectation = E((X - E(X, G)) * (Y - E(Y, G)), G)

print(covar, "\n")
print(expectation)

Random variable 'cov(X, Y|G)':
        cov(X, Y|G)
sample             
0        -28.494132
1        -28.494132
2          1.182969
3          1.182969
4          1.182969 

Random variable 'E(((X-E(X|G))*(Y-E(Y|G)))|G)':
        E(((X-E(X|G))*(Y-E(Y|G)))|G)
sample                              
0                         -28.494132
1                         -28.494132
2                           1.182969
3                           1.182969
4                           1.182969


#### Bilinearity and symmetry

The covariance is *symmetric*, meaning that

$$
\sigma(X,Y\mid \mathcal{G}) = \sigma(Y,X \mid \mathcal{G}).
$$

It is also *bilinear* which, in the presence of symmetry, means that

$$
\sigma(aX + Y,Z \mid \mathcal{G}) = a\sigma(X,Z \mid \mathcal{G}) + \sigma(Y,Z\mid \mathcal{G}),
$$

for all scalars $a$. We check these properties in the two cells.

In [35]:
print(cov(X, Y, G), "\n")
print(cov(Y, X, G))

Random variable 'cov(X, Y|G)':
        cov(X, Y|G)
sample             
0        -28.494132
1        -28.494132
2          1.182969
3          1.182969
4          1.182969 

Random variable 'cov(Y, X|G)':
        cov(Y, X|G)
sample             
0        -28.494132
1        -28.494132
2          1.182969
3          1.182969
4          1.182969


In [36]:
a = 3
Z = RandomVariable(domain=Omega, name="Z").from_randint(
    low=-20,
    high=21,
    random_state=rng,
)
Z.prob_measure = P

bilinear_lhs = cov(a * X + Y, Z, G)
bilinear_rhs = a * cov(X, Z, G) + cov(Y, Z, G)

print(bilinear_lhs, "\n")
print(bilinear_rhs)

Random variable 'cov(((3*X)+Y), Z|G)':
        cov(((3*X)+Y), Z|G)
sample                     
0                -63.736875
1                -63.736875
2                 50.557495
3                 50.557495
4                 50.557495 

Random variable '((3*cov(X, Z|G))+cov(Y, Z|G))':
        ((3*cov(X, Z|G))+cov(Y, Z|G))
sample                               
0                          -63.736875
1                          -63.736875
2                           50.557495
3                           50.557495
4                           50.557495
